# 13 — Cross-encoder reranking

> **Run order.** This notebook is step 13 of the pipeline. Earlier steps must
> have run at least once. See [`notebooks/README.md`](README.md).
>
> All reusable logic lives in `src/analyst/` — the package is unit-tested and
> type-checked, and these notebooks orchestrate it and show the results.

Everything so far embeds the question and the passage **separately** and compares
the two vectors — a bi-encoder. It never sees the pair side by side.

A **cross-encoder** reads the question and passage *together* and scores the
pair. It is far more accurate and far too slow to run over 9,982 chunks, so it
only ever reorders a shortlist.

**Which makes recall at the shortlist depth a hard ceiling.** This is why the
reranker was deliberately left until last: on the Day 3 baseline the ceiling at
depth 100 was 0.273, so a perfect reranker could not have exceeded that. Query
expansion moved it to 0.614, and that is what makes this step worth doing now.

In [1]:
from analyst.logging import configure_logging
configure_logging()

import pandas as pd
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from analyst import evaluation as ev
from analyst.config import get_settings
from analyst.embedding import RERANK_MODEL, Reranker
from analyst.retrievers import dense, hybrid, open_hybrid, open_store, reranked

MODEL = "bge-small"
DEPTH = 100   # shortlist size; recall@DEPTH of the inner retriever caps this

settings = get_settings()
questions = ev.load_questions(settings.data_dir / "benchmark" / "questions.jsonl")
embedder, store = open_store(settings, MODEL)
_, sparse, hstore = open_hybrid(settings, MODEL)
reranker = Reranker()
print(f"reranker: {RERANK_MODEL}   shortlist depth: {DEPTH}")

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.19.0 is incompatible with server version 1.12.4. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tumulu Preeyas\AppData\Local\Temp\fastembed_cache\models--Xenova--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Fetching 5 files:  20%|██        | 1/5 [00:00<00:02,  1.98it/s]

Fetching 5 files:  40%|████      | 2/5 [00:00<00:01,  2.89it/s]

Fetching 5 files:  80%|████████  | 4/5 [00:01<00:00,  4.38it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:11<00:00,  3.36s/it]

Fetching 5 files: 100%|██████████| 5/5 [00:11<00:00,  2.28s/it]

reranker: Xenova/ms-marco-MiniLM-L-6-v2   shortlist depth: 100

## Score both expanded retrievers, reranked

The reranker composes over any `SearchFn`, so this is the same wrapper applied
twice — no retrieval code changes.

In [2]:
inners = {
    "dense+expand": dense(embedder, store, "ticker+year", expand=True),
    "hybrid+expand": hybrid(embedder, sparse, hstore, "ticker+year", expand=True),
}

for name, inner in inners.items():
    label = f"{name}+rerank"
    search = reranked(inner, reranker, depth=DEPTH, expand=True)
    run = ev.build_run(
        ev.RunConfig(retriever=label, model=MODEL, filters="ticker+year",
                     limit=max(ev.K_VALUES), points=store.count()),
        ev.evaluate(questions, search, limit=max(ev.K_VALUES)),
        questions,
    )
    ev.append_run(run)
    m = run.metrics
    print(f"{label:<24} R@1 {m.recall_at[1]:.3f}  R@5 {m.recall_at[5]:.3f}  "
          f"R@10 {m.recall_at[10]:.3f}  MRR {m.mrr:.3f}  p50 {m.p50_ms:.0f} ms")

dense+expand+rerank      R@1 0.023  R@5 0.091  R@10 0.091  MRR 0.046  p50 5314 ms

hybrid+expand+rerank     R@1 0.023  R@5 0.091  R@10 0.091  MRR 0.046  p50 5381 ms

## Did reranking help?

Compared against each retriever's own un-reranked run, so the delta isolates the
reranker rather than mixing in the expansion gain.

In [3]:
ledger = ev.load_runs()
latest = {}
for r in ledger:
    if r.config.model == MODEL and r.config.filters == "ticker+year":
        latest[r.config.retriever] = r

order = ["dense", "hybrid", "dense+expand", "hybrid+expand",
         "dense+expand+rerank", "hybrid+expand+rerank"]
table = pd.DataFrame([latest[k].row() for k in order if k in latest])
print(table.to_string(index=False))

print()
for base_name in ("dense+expand", "hybrid+expand"):
    rr = f"{base_name}+rerank"
    if base_name in latest and rr in latest:
        b, a = latest[base_name].metrics, latest[rr].metrics
        print(f"{base_name} -> +rerank:  "
              f"R@5 {b.recall_at[5]:.3f} -> {a.recall_at[5]:.3f}   "
              f"R@10 {b.recall_at[10]:.3f} -> {a.recall_at[10]:.3f}   "
              f"MRR {b.mrr:.3f} -> {a.mrr:.3f}")

                                    run            retriever     model     filters    R@1    R@3    R@5   R@10    MRR   pR@5  points  p50_ms    bench           git
               dense-bge-small-02c4b4ed                dense bge-small ticker+year 0.0227 0.0455 0.0455 0.0909 0.0392 0.0909    9982    84.8 2c4aedf3 3e65907-dirty
              hybrid-bge-small-ee1a298f               hybrid bge-small ticker+year 0.0227 0.0682 0.0682 0.1136 0.0449 0.1136    9982    90.0 2c4aedf3 3e65907-dirty
        dense+expand-bge-small-b60d0b39         dense+expand bge-small ticker+year 0.0455 0.0455 0.0682 0.1136 0.0559 0.1364    9982    87.7 2c4aedf3 2f3c9a3-dirty
       hybrid+expand-bge-small-6510b044        hybrid+expand bge-small ticker+year 0.0455 0.0455 0.0682 0.0909 0.0544 0.1136    9982    91.1 2c4aedf3 2f3c9a3-dirty
 dense+expand+rerank-bge-small-b261fe8f  dense+expand+rerank bge-small ticker+year 0.0227 0.0682 0.0909 0.0909 0.0462 0.1364    9982  5313.6 2c4aedf3 2f3c9a3-dirty
hybrid+expand+re

dense+expand -> +rerank:  R@5 0.068 -> 0.091   R@10 0.114 -> 0.091   MRR 0.056 -> 0.046

hybrid+expand -> +rerank:  R@5 0.068 -> 0.091   R@10 0.091 -> 0.091   MRR 0.054 -> 0.046

## The cost

The reranker runs the cross-encoder over `DEPTH` passages per question, so its
latency is the honest price of the accuracy. Compare `p50_ms` above against the
un-reranked rows: this is the number that decides whether it ships.

In [4]:
cost = pd.DataFrame([
    {"retriever": k, "p50_ms": latest[k].metrics.p50_ms, "R@5": latest[k].metrics.recall_at[5]}
    for k in order if k in latest
])
print(cost.to_string(index=False))

           retriever  p50_ms    R@5
               dense    84.8 0.0455
              hybrid    90.0 0.0682
        dense+expand    87.7 0.0682
       hybrid+expand    91.1 0.0682
 dense+expand+rerank  5313.6 0.0909
hybrid+expand+rerank  5380.9 0.0909

## The ledger

In [5]:
print(ev.write_leaderboard(ev.load_runs()))

C:\Internships\SELF_PROJECTS\MULTIMODAL_AGENTIC_RAG\results\leaderboard.md